### How the data is organised and accessed

We have acquired three datasets from UN Comtrade for this project. They cover Nigeria's reported imports and exports between 2016 and 2025.

| Dataset | What it contains | Python variable |
|---|---|---|
| **Total import–export data** | Nigeria's total imports and exports by trading partner, without breaking them down into individual commodities | `total_trade` |
| **Commodity imports** | Commodities Nigeria imported, the countries they came from, and their reported trade values and other available measurements | `commodity_imports` |
| **Commodity exports** | Commodities Nigeria exported, the destination countries, and their reported trade values and other available measurements | `commodity_exports` |

All three datasets have been uploaded to our **shared Google Drive folder**, which serves as the central storage location for the project.

**You do not need to download or upload the CSV files yourself.** The notebook already contains the Google Drive file IDs and the code needed to retrieve them.

Here's how it works:

1. **Run the setup cell first.** This installs the required package (`gdown`), imports the Python libraries and defines the `load_trade_csv()` function.
2. **Run the dataset-loading cells.** Each cell uses its file ID to retrieve the corresponding CSV from Google Drive and load it into Colab.
3. **Start your analysis.** Once the cells finish running, the three datasets will be available as `total_trade`, `commodity_imports` and `commodity_exports`.

The `load_trade_csv()` function also checks whether a file has already been downloaded into the current Colab environment. If it exists, the function uses that copy instead of downloading it again.

**A few things to remember:**

- Google Colab uses temporary storage. If your runtime resets, simply rerun the setup and loading cells to retrieve the files again.
- The original CSV files remain in our shared Google Drive folder. Changes you make to the DataFrames in Colab will not automatically change those original files.
- The shared folder is our central source for the datasets, while Colab is where we load, inspect, clean and analyse them.
- The datasets contain commodity descriptions and classification information. See the acquisition notes below for guidance on interpreting the columns.
- **Saving updated datasets:** If you clean, transform or otherwise update a dataset, save the new CSV in our shared Google Drive folder. Give it a clear filename so others can distinguish it from the original. Then add a simple loading cell to the notebook, using the same `load_trade_csv()` function, so everyone can access the updated version without uploading it manually.
- **Preserve the originals:** Do not overwrite the original datasets. Keeping both the original and updated versions will allow us to trace changes, compare results and reproduce our work.

This arrangement allows everyone to work with the same source files without having to repeat the acquisition process.

In [ ]:
# 1.1 Setup and data access
# Data acquisition: shared dependencies
!pip -q install gdown

import pandas as pd
import gdown
from pathlib import Path

# Shared dataset location in the Colab runtime.
DATA_DIR = Path("/content/trade_data")
DATA_DIR.mkdir(exist_ok=True)

# Retrieve a dataset from the team's shared Google Drive.
def load_trade_csv(file_id, filename):
    file_path = DATA_DIR / filename

    if not file_path.exists():
        gdown.download(id=file_id, output=str(file_path), quiet=False)

    return pd.read_csv(
        file_path,
        index_col=False,
        low_memory=False,
        dtype={"cmdCode": str}
    )


In [ ]:

# Dataset 1: Nigeria's annual total trade by partner, 2016–2025.
# Includes imports, exports and the World aggregate.

total_trade = load_trade_csv(
    file_id="1FKwgVLJd0tZwRePHItAkkXCZMT0nAF7_",
    filename="un_comtrade_nigeria_2016_2025.csv"
)

# Verify the acquired dataset's structure and scope.
print("Shape:", total_trade.shape)
print("Years:", sorted(total_trade["refYear"].unique()))
print("Flows:", total_trade["flowDesc"].unique())
print("Commodity codes:", total_trade["cmdCode"].unique())

display(total_trade.head(3))


# Keep file_path aligned with the dataset being loaded.
# Colab retains variables from previously executed cells.

In [ ]:

# Dataset 2: Nigeria's commodity-by-partner trade, 2016–2025.
# HS4 commodities, imports and exports.

commodity_imports = load_trade_csv(
    file_id="1czU2t2csZ-_s5vuC8zNVzYUzC_cy8xoV",
    filename="commodity_imports_combined.csv"
)

# Verify the file loaded and the columns are aligned.
print("Rows and columns:", commodity_imports.shape)

display(commodity_imports[[
    "refYear",
    "reporterDesc",
    "flowDesc",
    "partnerDesc",
    "classificationCode",
    "cmdCode",
    "cmdDesc",
    "primaryValue"
]].head(5))


In [ ]:

# Acquisition verification only, not exploratory analysis.
print("Years:", sorted(commodity_imports["refYear"].dropna().unique().tolist()))
print("Trade flows:", commodity_imports["flowDesc"].dropna().unique().tolist())
print("Reporter:", commodity_imports["reporterDesc"].dropna().unique().tolist())
print("HS classifications:", commodity_imports["classificationCode"].dropna().unique().tolist())
print("Commodity code lengths:",
      sorted(commodity_imports["cmdCode"].dropna().astype(str).str.len().unique().tolist()))


In [ ]:
# Dataset 3: Nigeria's commodity-by-partner trade, 2016–2025.
# HS4 commodities, imports and exports.

commodity_exports = load_trade_csv(
    file_id="1te8mOvkyGiHSi3-pF_KmZY9yREO1GehX",
    filename="commodity_exports_2016_2025"
)

# Verify the file loaded and the columns are aligned.
print("Rows and columns:", commodity_exports.shape)

display(commodity_exports[[
    "refYear",
    "reporterDesc",
    "flowDesc",
    "partnerDesc",
    "classificationCode",
    "cmdCode",
    "cmdDesc",
    "primaryValue"
  ]].head(5))

In [ ]:
# Acquisition verification.
print("Years:", sorted(commodity_exports["refYear"].dropna().unique().tolist()))
print("Trade flows:", commodity_exports["flowDesc"].dropna().unique().tolist())
print("Reporter:", commodity_exports["reporterDesc"].dropna().unique().tolist())
print("HS classifications:", commodity_exports["classificationCode"].dropna().unique().tolist())
print("Commodity code lengths:",
      sorted(commodity_exports["cmdCode"].dropna().astype(str).str.len().unique().tolist()))

### Commodity-level trade data: acquisition summary

**Source:** UN Comtrade  
**Website:** https://comtradeplus.un.org/

We downloaded Nigeria's import and export data from 2016 to 2025, covering individual commodities and the countries Nigeria trades with.

**Files acquired:**

- **Commodity imports:** Three CSV files covering different periods were downloaded and combined into `commodity_imports_combined.csv`. The final dataset contains **216,371 rows and 47 columns**.
- **Commodity exports:** We were able to download all ten years at once. The file contains **16,848 rows and 47 columns**.

We downloaded the import data in batches because UN Comtrade has a 100,000-record download limit. Splitting the years helped us avoid losing records to that limit.

**A few things to know before analysing the data:**

- **Commodity codes:** The `cmdCode` column contains the standard Harmonised System (HS) code used to identify a commodity. The `cmdDesc` column tells you what that commodity is. For example, `1006` represents rice.
- **Commodity classifications:** The data contains two HS classification versions, `H4` and `H6`. Some commodity definitions may have changed between versions, so be careful when comparing the same commodity across years.
- **Trading partners:** The `partnerDesc` column identifies the country Nigeria traded with. You will also find records labelled `World`. This is an aggregate representing trade with the world, not another country. **Do not add the World figures to those of individual countries**, as this would double-count trade.
- **Commodity codes as text:** Keep `cmdCode` as text rather than a number. Otherwise, codes beginning with zero may lose their first digit.
- **Record counts:** More rows do not necessarily mean more trade. Each row represents a reported trade record for a particular combination of year, commodity and trading partner, not an individual shipment.

The original data has been preserved. No cleaning or exploratory analysis has been carried out in this section.

**Note:** The datasets cover all ten years, but the completeness of individual records and any possible duplication from combining the import files still need to be checked.

## Phase 1. Protect and profile the source data

**Research link:** Objective 1 and research question 1.

We begin by preserving untouched DataFrame copies, attaching source-row tracing information, profiling field completeness and initialising an audit log. These steps help distinguish observations about the source data from transformations made during analysis.

**Outputs:** raw working copies, profiling tables and an analytical audit log. The original files in shared Google Drive are not overwritten.


In [ ]:
# 1. Create untouched raw copies of the datasets
total_trade_raw = total_trade.copy()
commodity_imports_raw = commodity_imports.copy()
commodity_exports_raw = commodity_exports.copy()

# 2. Add tracing metadata columns to each raw dataset
total_trade_raw['source_file'] = 'un_comtrade_nigeria_2016_2025.csv'
total_trade_raw['source_row'] = total_trade_raw.index

commodity_imports_raw['source_file'] = 'commodity_imports_combined.csv'
commodity_imports_raw['source_row'] = commodity_imports_raw.index

commodity_exports_raw['source_file'] = 'commodity_exports_2016_2025'
commodity_exports_raw['source_row'] = commodity_exports_raw.index

print("Untouched raw copies instantiated with 'source_file' and 'source_row' metadata columns successfully.")

In [ ]:
# 3. Generate profiling summaries for each raw dataset
def profile_dataset(df, name):
    profile = pd.DataFrame({
        'Data Type': df.dtypes,
        'Non-Null Count': df.notnull().sum(),
        'Null Count': df.isnull().sum(),
        'Null (%)': (df.isnull().mean() * 100).round(2)
    })
    print(f"\n{'='*15} PROFILE FOR: {name} ({df.shape[0]} rows, {df.shape[1]} columns) {'='*15}")
    display(profile.head(10))
    return profile

profile_total = profile_dataset(total_trade_raw, 'total_trade_raw')
profile_imports = profile_dataset(commodity_imports_raw, 'commodity_imports_raw')
profile_exports = profile_dataset(commodity_exports_raw, 'commodity_exports_raw')

In [ ]:
# 4. Initialize the analytical audit/action log DataFrame
audit_log = pd.DataFrame(columns=[
    'Step', 'Action', 'Reason', 'Rows Affected', 'Output File'
])

# Log our initialization and baseline creation
init_entry = {
    'Step': '1.0',
    'Action': 'Instantiate Raw Baselines & Profile',
    'Reason': 'Preserve original data state and define tracing indexes before applying transformations.',
    'Rows Affected': f"total_trade: {len(total_trade_raw)}, imports: {len(commodity_imports_raw)}, exports: {len(commodity_exports_raw)}",
    'Output File': 'Memory Variables'
}

audit_log = pd.concat([audit_log, pd.DataFrame([init_entry])], ignore_index=True)
display(audit_log)

## Phase 2. Understand the fields and their meanings

**Research link:** Objective 1 and research question 1.

This phase compares the import and export schemas, checks fields needed for analysis, documents the meaning of important UN Comtrade columns and converts identifier fields to suitable types. In particular, `cmdCode` remains text so HS codes such as `0303` retain their leading zeros.

**Interpretation:** H4 and H6 refer to classification revisions. They should not be interpreted as the number of digits in the selected commodity code.


In [ ]:
# 1. Verify if both datasets have the same columns
import_cols = set(commodity_imports_raw.columns)
export_cols = set(commodity_exports_raw.columns)

print(f"Imports columns count: {len(import_cols)}")
print(f"Exports columns count: {len(export_cols)}")
print(f"Are columns identical?: {import_cols == export_cols}")
if import_cols != export_cols:
    print("Difference in Imports:", import_cols - export_cols)
    print("Difference in Exports:", export_cols - import_cols)

In [ ]:
# 2. Check for the existence of key analytical fields
required_fields = {
    'Temporal': ['refYear', 'period', 'freqCode'],
    'Geography/Flow': ['reporterCode', 'reporterDesc', 'partnerCode', 'partnerDesc', 'flowCode', 'flowDesc'],
    'Commodity': ['cmdCode', 'cmdDesc', 'classificationCode', 'aggrLevel', 'isLeaf'],
    'Breakdowns': ['motCode', 'customsCode', 'partner2Code'],
    'Value': ['primaryValue', 'cifvalue', 'fobvalue'],
    'Quantities': ['qty', 'qtyUnitCode', 'netWgt'],
    'Flags': ['isAggregate', 'isReported', 'legacyEstimationFlag']
}

print("--- Column Checklist --- ")
for category, fields in required_fields.items():
    print(f"\nCategory: {category}")
    for f in fields:
        exists = f in commodity_imports_raw.columns
        status = "Present" if exists else "MISSING"
        print(f"  - {f}: {status}")

In [ ]:
# 3. Cast correct data types securely (especially cmdCode as a string)
STR_COLS = ['cmdCode', 'partnerDesc', 'reporterDesc', 'flowDesc',
            'classificationCode', 'flowCode']
NUM_COLS = ['primaryValue', 'cifvalue', 'fobvalue', 'qty', 'netWgt']

def cast_trade_types(df):
    out = df.copy()
    for col in STR_COLS:
        if col in out.columns:
            # "string" dtype keeps real missing values as <NA>.
            # astype(str) would turn them into the text "nan".
            out[col] = out[col].astype("string").str.strip()
            out[col] = out[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    for col in NUM_COLS:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")   # missing stays missing, never 0
    return out

commodity_imports_typed = cast_trade_types(commodity_imports_raw)
commodity_exports_typed = cast_trade_types(commodity_exports_raw)

entry = {
    'Step': '2.0',
    'Action': 'Type Casting & Verification',
    'Reason': 'Standardize cmdCode as a string to protect leading zeros, and verify presence of value and metric columns.',
    'Rows Affected': f"imports: {len(commodity_imports_typed)}, exports: {len(commodity_exports_typed)}",
    'Output File': 'Memory Variables'
}
audit_log = pd.concat([audit_log, pd.DataFrame([entry])], ignore_index=True)
print("Data type casting successfully completed!")

In [ ]:
for name, d in [("Imports", commodity_imports_typed), ("Exports", commodity_exports_typed)]:
    print(f"\n--- {name} ---")
    for col in STR_COLS:
        if col in d.columns:
            n_missing = int(d[col].isna().sum())
            n_nan_text = int((d[col].str.lower() == "nan").sum())
            print(f"{col:20s} missing: {n_missing:>8,}   text 'nan': {n_nan_text:>6,}")
    print("cmdCode lengths:", d["cmdCode"].str.len().value_counts(dropna=False).to_dict())

### Data dictionary: fields used in this study

| Field | Meaning in the notebook | Why it matters |
|---|---|---|
| `refYear`, `period` | Reference year and reporting period | Coverage and comparisons across years |
| `reporterCode`, `reporterDesc` | Reporting economy | Verify Nigeria as reporter |
| `flowDesc` | Import or export flow | Keep directions separate |
| `partnerCode`, `partnerDesc` | Partner identifier and label | Country rankings and aggregate exclusion |
| `cmdCode`, `cmdDesc` | HS commodity code and description | Commodity selection and comparisons |
| `classificationCode` / classification field | HS classification revision | Assess comparability across periods |
| `primaryValue` | Main reported trade value in USD | Trade-value calculations |
| `netWgt`, `qty` | Reported weight and quantity | Unit-value and quantity screening |
| `cifvalue`, `fobvalue` | Additional value fields | Completeness diagnostics; not assumed interchangeable |

**Important distinction:** `World` is an aggregate, not a country. Do not add World records to country-partner totals. A missing auxiliary field does not automatically mean that the main trade value is missing.


In [ ]:
def check_value_basis(df, flow_label, name):
    sub = df[df["flowDesc"].str.lower() == flow_label.lower()]
    print(f"\n{name}: {len(sub):,} {flow_label} rows")
    for col in ["cifvalue", "fobvalue"]:
        if col not in sub.columns:
            print(f"  {col}: column not in file")
            continue
        has = sub[col].notna() & (sub[col] > 0)
        n = int(has.sum())
        same = int(((sub.loc[has, "primaryValue"] - sub.loc[has, col]).abs() <= 1).sum()) if n else 0
        print(f"  {col}: filled on {n:,} rows ({n / max(len(sub), 1) * 100:.1f}%); "
              f"primaryValue equals it on {same:,} of those")

check_value_basis(commodity_imports_typed, "Import", "Imports file")
check_value_basis(commodity_exports_typed, "Export", "Exports file")

## Phase 3. Diagnose quality issues in the source records

**Research link:** Objectives 1–2 and research questions 1–2.

The checks examine missing values, duplicate records, reporting years, partner labels, commodity classification versions, trade values and quantity/weight availability. They also compare World aggregates with partner-level records.

A **flag** means that a record merits investigation. It is not confirmation of an error. Diagnostic outputs are preserved for later comparison and for the issue register.


In [ ]:
import re
import numpy as np

MANUAL_NON_COUNTRY = []   # exact partnerDesc values you decide are NOT single countries (after reviewing C11)
MANUAL_COUNTRY = []       # exact partnerDesc values the rules flagged wrongly

def is_non_country(desc, code=None):
    """True for World, 'nes' groups, unspecified areas, free zones and similar. False for single countries."""
    if desc is None or pd.isna(desc):
        return True                      # a missing partner is not a country; it shows up in the review table
    if desc in MANUAL_COUNTRY:
        return False
    if desc in MANUAL_NON_COUNTRY:
        return True
    if code is not None and pd.notna(code) and pd.to_numeric(code, errors="coerce") == 0:
        return True                      # partner code 0 is World
    d = str(desc).strip().lower()
    if d == "world":
        return True
    if re.search(r"\bnes\b", d):         # whole word only: "Other Asia, nes" yes, "Indonesia" no
        return True
    return any(k in d for k in ["unspecified", "free zone", "special categor",
                                "bunker", "ship stores", "not elsewhere"])

def partner_review(df, name):
    t = (df.groupby(["partnerCode", "partnerDesc"], dropna=False)["primaryValue"]
           .agg(rows="size", value="sum").reset_index())
    t["flagged_non_country"] = [is_non_country(d, c) for d, c in zip(t["partnerDesc"], t["partnerCode"])]
    is_world = pd.to_numeric(t["partnerCode"], errors="coerce") == 0
    non_world_total = t.loc[~is_world, "value"].sum()
    t["share_of_non_world_value_%"] = (t["value"] / non_world_total * 100).round(2)
    t.loc[is_world, "share_of_non_world_value_%"] = np.nan
    t = t.sort_values(["flagged_non_country", "value"], ascending=[False, False])
    print(f"\n=== PARTNER REVIEW: {name} ===")
    print("FLAGGED = treated as NOT a single country. Check this list by eye.")
    display(t[t["flagged_non_country"]])
    print("NOT flagged (top 15 by value). Scan the full list below for anything that is not a country:")
    display(t[~t["flagged_non_country"]].head(15))
    print(sorted(t.loc[~t["flagged_non_country"], "partnerDesc"].dropna().tolist()))
    return t

def non_country_share_by_year(df, name):
    d = df[pd.to_numeric(df["partnerCode"], errors="coerce") != 0].copy()   # drop the World row first
    d["non_country"] = [is_non_country(x, c) for x, c in zip(d["partnerDesc"], d["partnerCode"])]
    s = d.groupby(["refYear", "non_country"])["primaryValue"].sum().unstack(fill_value=0)
    s = s.reindex(columns=[False, True], fill_value=0)
    s["non_country_share_%"] = (s[True] / (s[True] + s[False]) * 100).round(2)
    print(f"\nShare of value held by non-country partners, by year ({name}):")
    display(s)

def hs4_rows(df):
    """Keep one commodity level only (4-digit), so levels are never added together."""
    out = df
    if "aggrLevel" in out.columns:
        out = out[pd.to_numeric(out["aggrLevel"], errors="coerce") == 4]
    return out[out["cmdCode"].str.len() == 4]

In [ ]:
# 9. Count missing values and percentages, and check missingness over years for key fields
def check_missingness(df, name):
    missing_df = pd.DataFrame({
        'Missing Count': df.isnull().sum(),
        'Missing %': (df.isnull().mean() * 100).round(2)
    }).sort_values(by='Missing %', ascending=False)
    print(f"\n--- Missingness in {name} (Top 10 columns) ---")
    display(missing_df.head(10))

    # Yearly missingness check for critical value fields
    print(f"Yearly Missingness for primaryValue in {name}:")
    display(df.groupby('refYear')['primaryValue'].apply(lambda x: x.isnull().sum()))

check_missingness(commodity_imports_typed, 'Imports')
check_missingness(commodity_exports_typed, 'Exports')

### Visual 1. Missingness in key fields

**RQ1:** Percentage of missing values by field and flow. Missing auxiliary fields are shown separately from the primary trade value. Denominator: rows in each typed commodity dataset.

In [ ]:
import matplotlib.pyplot as plt
fields = [c for c in ['primaryValue','netWgt','qty','cifvalue','fobvalue'] if c in commodity_imports_typed.columns and c in commodity_exports_typed.columns]
missing_plot = pd.DataFrame({
    'Imports': commodity_imports_typed[fields].isna().mean().mul(100),
    'Exports': commodity_exports_typed[fields].isna().mean().mul(100)
})
ax = missing_plot.plot(kind='bar', figsize=(10, 4.5), rot=0)
ax.set(title='Missing values in selected trade fields', ylabel='Missing records (%)', xlabel='Field', ylim=(0, 105))
ax.legend(title='Trade flow')
plt.tight_layout(); plt.show()
display(missing_plot.round(2).rename_axis('Field'))


In [ ]:
# 10. Check exact duplicate rows
print(f"Exact duplicated rows in Imports (excluding tracking metadata): {commodity_imports_typed.drop(columns=['source_file', 'source_row']).duplicated().sum()}")
print(f"Exact duplicated rows in Exports (excluding tracking metadata): {commodity_exports_typed.drop(columns=['source_file', 'source_row']).duplicated().sum()}")
print(f"Exact duplicated rows in Total Trade (excluding tracking metadata): {total_trade_raw.drop(columns=['source_file', 'source_row']).duplicated().sum()}")

In [ ]:
# 11 and 12. Key checks
STRUCTURE_COLS = ['freqCode', 'typeCode', 'motCode', 'customsCode', 'partner2Code',
                  'aggrLevel', 'isLeaf', 'isAggregate', 'isReported',
                  'flowDesc', 'classificationCode']

def structure_report(df, name):
    print(f"\n=========== STRUCTURE: {name} ===========")
    for col in STRUCTURE_COLS:
        if col in df.columns:
            print(f"\n{col}:")
            print(df[col].value_counts(dropna=False).head(10).to_string())
        else:
            print(f"\n{col}: column not in file")

def build_analysis_mask(df, name):
    """Keeps only total-level, annual, HS4 rows. Prints how many rows each rule removes."""
    rules = {'freqCode': ['A'], 'motCode': ['0'], 'customsCode': ['C00'],
             'partner2Code': ['0'], 'aggrLevel': ['4']}
    mask = pd.Series(True, index=df.index)
    for col, ok in rules.items():
        if col not in df.columns:
            print(f"{name}: {col} not in file, rule skipped")
            continue
        vals = df[col].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
        keep = vals.isin(ok).fillna(False)
        removed = int((mask & ~keep).sum())
        if removed == len(df):
            print(f"{name}: WARNING {col} rule would remove ALL rows (values are {vals.unique()[:5]}). Rule skipped. Check the UN documentation.")
            continue
        print(f"{name}: {col} must be in {ok} -> removes {removed:,} rows")
        mask &= keep
    print(f"{name}: keeps {int(mask.sum()):,} of {len(df):,} rows")
    return mask

def key_check(df, mask, name):
    key = [c for c in ['refYear', 'reporterCode', 'partnerCode', 'flowCode', 'cmdCode',
                       'classificationCode', 'freqCode', 'motCode', 'customsCode', 'partner2Code']
           if c in df.columns]
    f = df[mask]
    dup = f[f.duplicated(key, keep=False)].sort_values(key)
    print(f"{name}: key = {key}")
    print(f"{name}: rows sharing a key after the filter = {len(dup):,} (expect 0)")
    if len(dup):
        display(dup.head(10))
    return dup

structure_report(commodity_imports_typed, "Imports")
structure_report(commodity_exports_typed, "Exports")
imp_mask = build_analysis_mask(commodity_imports_typed, "Imports")
exp_mask = build_analysis_mask(commodity_exports_typed, "Exports")
dup_imp = key_check(commodity_imports_typed, imp_mask, "Imports")
dup_exp = key_check(commodity_exports_typed, exp_mask, "Exports")

In [ ]:
# 13. Years present, partner and commodity variety metrics by year
def year_coverage(df, name):
    d = df[pd.to_numeric(df["partnerCode"], errors="coerce") != 0]   # partner rows only
    g = d.groupby("refYear").agg(rows=("primaryValue", "size"),
                                 partners=("partnerDesc", "nunique"),
                                 commodities=("cmdCode", "nunique"),
                                 partner_row_value=("primaryValue", "sum"))
    for col in ["rows", "partners", "commodities"]:
        g[f"{col}_%_of_median"] = (g[col] / g[col].median() * 100).round(0)
    g["candidate_partial"] = ((g["rows_%_of_median"] < 80) |
                              (g["partners_%_of_median"] < 80) |
                              (g["commodities_%_of_median"] < 80))
    print(f"\n--- Coverage by year ({name}) ---")
    display(g)
    return g

yc_imp = year_coverage(commodity_imports_typed, "Imports")
yc_exp = year_coverage(commodity_exports_typed, "Exports")
print("Revisions: not assessed.")

In [ ]:
PARTIAL_YEARS = []

audit_log = pd.concat([audit_log, pd.DataFrame([{
    'Step': '3.9',
    'Action': 'Decide partial years: none',
    'Reason': 'Imports: rows 89-108%, partners 92-102%, commodities 99-101% of the median in every year, including 2025. Exports: fewer commodities in 2016-2020 (60-91% of median) with partners steady (91-97%); gradual rise treated as a coverage finding, not partial years.',
    'Rows Affected': 'n/a',
    'Output File': 'Diagnostic Summaries Only'}])], ignore_index=True)

if PARTIAL_YEARS is None:
    raise ValueError("Fill in PARTIAL_YEARS (see the C09 tables), then run this cell again.")

ALL_YEARS = sorted(commodity_imports_typed["refYear"].dropna().astype(int).unique().tolist())
COMPLETE_YEARS = [y for y in ALL_YEARS if y not in PARTIAL_YEARS]
HIST_YEARS = COMPLETE_YEARS           # Scenario A uses all complete years
LATEST_3 = COMPLETE_YEARS[-3:]        # Scenario B and commodity selection use the latest three complete years
LATEST_YEAR = COMPLETE_YEARS[-1]

print("Partial years:", PARTIAL_YEARS)
print("Complete years:", COMPLETE_YEARS)
print("Latest three complete years:", LATEST_3, "| latest complete year:", LATEST_YEAR)

def add_year_flag(df):
    out = df.copy()
    out["is_partial_year"] = out["refYear"].isin(PARTIAL_YEARS)
    return out

In [ ]:
partner_review_imp = partner_review(commodity_imports_typed, "Imports")
partner_review_exp = partner_review(commodity_exports_typed, "Exports")
non_country_share_by_year(commodity_imports_typed, "Imports")
non_country_share_by_year(commodity_exports_typed, "Exports")

In [ ]:
# 15. Check distribution of HS versions across years
print("--- HS Classification Distribution by Year (Imports) ---")
display(commodity_imports_typed.groupby(['refYear', 'classificationCode']).size().unstack(fill_value=0))

print("\n--- HS Classification Distribution by Year (Exports) ---")
display(commodity_exports_typed.groupby(['refYear', 'classificationCode']).size().unstack(fill_value=0))

In [ ]:
# 16. Statistics on primaryValue, zero/negative counts, and source references
def value_metrics(df, name):
    non_null_val = df['primaryValue'].dropna()
    print(f"\n--- Stats for primaryValue in {name} ---")
    print(f"Min: {non_null_val.min()} | Max: {non_null_val.max()} | Median: {non_null_val.median()}")

    zero_rows = df[df['primaryValue'] == 0]['source_row'].tolist()
    neg_rows = df[df['primaryValue'] < 0]['source_row'].tolist()

    print(f"Count of zero values: {len(zero_rows)} (First 5 source_rows: {zero_rows[:5]})")
    print(f"Count of negative values: {len(neg_rows)} (First 5 source_rows: {neg_rows[:5]})")

value_metrics(commodity_imports_typed, 'Imports')
value_metrics(commodity_exports_typed, 'Exports')

In [ ]:
# 17. Quantities check: primaryValue is positive but volume/weight is missing/zero
def check_quantities(df, name):
    print(f"\n--- Quantity/Weight Checks for {name} ---")
    positive_val = df[df['primaryValue'] > 0]
    zero_or_missing_qty = positive_val[(positive_val['qty'].isnull()) | (positive_val['qty'] == 0)]
    zero_or_missing_wgt = positive_val[(positive_val['netWgt'].isnull()) | (positive_val['netWgt'] == 0)]

    print(f"Positive value rows with zero/null qty: {len(zero_or_missing_qty)} ({round(len(zero_or_missing_qty)/len(positive_val)*100, 2)}%)")
    print(f"Positive value rows with zero/null netWgt: {len(zero_or_missing_wgt)} ({round(len(zero_or_missing_wgt)/len(positive_val)*100, 2)}%)")
    print("Unique Quantity Units used:", df['qtyUnitCode'].unique())

check_quantities(commodity_imports_typed, 'Imports')
check_quantities(commodity_exports_typed, 'Exports')

In [ ]:
# 19 and 20
def world_vs_partners(df, name):
    d = hs4_rows(df)
    is_world = pd.to_numeric(d["partnerCode"], errors="coerce") == 0
    out = pd.DataFrame({
        "world_row_total": d[is_world].groupby("refYear")["primaryValue"].sum(),
        "sum_of_partner_rows": d[~is_world].groupby("refYear")["primaryValue"].sum(),
    })
    out["difference"] = out["world_row_total"] - out["sum_of_partner_rows"]
    out["difference_%_of_world"] = (out["difference"] / out["world_row_total"] * 100).round(2)
    print(f"\n--- World row vs sum of partner rows, HS4 only ({name}) ---")
    display(out.round(2))
    return out

def total_vs_commodity(total_df, cmd_df, flow_label, name):
    t = total_df[total_df["flowDesc"].str.lower() == flow_label.lower()]
    if t["cmdCode"].str.upper().eq("TOTAL").any():
        t = t[t["cmdCode"].str.upper() == "TOTAL"]
    c = hs4_rows(cmd_df)
    t_world = pd.to_numeric(t["partnerCode"], errors="coerce") == 0
    c_world = pd.to_numeric(c["partnerCode"], errors="coerce") == 0
    out = pd.DataFrame({
        "total_trade_World": t[t_world].groupby("refYear")["primaryValue"].sum(),
        "commodity_World_rows": c[c_world].groupby("refYear")["primaryValue"].sum(),
        "total_trade_partners": t[~t_world].groupby("refYear")["primaryValue"].sum(),
        "commodity_partners": c[~c_world].groupby("refYear")["primaryValue"].sum(),
    })
    out["World_diff_%"] = ((out["total_trade_World"] - out["commodity_World_rows"]) / out["total_trade_World"] * 100).round(2)
    out["Partner_diff_%"] = ((out["total_trade_partners"] - out["commodity_partners"]) / out["total_trade_partners"] * 100).round(2)
    print(f"\n--- total_trade vs commodity file ({name}) ---")
    display(out.round(2))
    return out

wp_imp = world_vs_partners(commodity_imports_typed, "Imports")
wp_exp = world_vs_partners(commodity_exports_typed, "Exports")
tc_imp = total_vs_commodity(total_trade_raw, commodity_imports_typed, "Import", "Imports")
tc_exp = total_vs_commodity(total_trade_raw, commodity_exports_typed, "Export", "Exports")

In [ ]:
# 21. Create and output Issue Register v1 as a DataFrame
issue_register = pd.DataFrame(columns=[
    'ID', 'Dataset', 'Quality Dimension', 'Check', 'Commodity', 'Finding', 'Evidence/Source_Row', 'Status'
])

# Populate baseline issues found programmatically
issues_list = [
    {
        'ID': 'ISS-001', 'Dataset': 'Imports/Exports', 'Quality Dimension': 'Completeness',
        'Check': 'Check missing values in columns', 'Commodity': 'ALL',
        'Finding': 'cifvalue and fobvalue have > 75% missingness. However primaryValue is 100% complete.',
        'Evidence/Source_Row': 'profile_imports / profile_exports DataFrames', 'Status': 'Confirmed problem'
    },
    {
        'ID': 'ISS-002', 'Dataset': 'Imports', 'Quality Dimension': 'Uniqueness',
        'Check': 'Check duplicate rows', 'Commodity': 'ALL',
        'Finding': 'Zero exact row duplicates found in raw imports when excluding source_file and source_row metadata.',
        'Evidence/Source_Row': 'None', 'Status': 'No problem found'
    },
    {
        'ID': 'ISS-003', 'Dataset': 'Imports/Exports', 'Quality Dimension': 'Consistency',
        'Check': 'Non-individual partners', 'Commodity': 'ALL',
        'Finding': 'WORLD and NES regional designations are grouped in partnerDesc.',
        'Evidence/Source_Row': "['WORLD', 'OTHER ASIA, NES', 'AREAS, NES']", 'Status': 'Confirmed problem'
    },
    {
        'ID': 'ISS-004', 'Dataset': 'Imports', 'Quality Dimension': 'Completeness',
        'Check': 'Value is positive but quantity is zero/null', 'Commodity': 'ALL',
        'Finding': 'A segment of imports has positive primaryValue but null/zero quantities.',
        'Evidence/Source_Row': 'qty column value inspection', 'Status': 'Confirmed problem'
    }
]

issue_register = pd.concat([issue_register, pd.DataFrame(issues_list)], ignore_index=True)
display(issue_register)

## Phase 4. Prepare analytical working copies

**Research link:** Objectives 1–3 and research questions 1–3.

Working copies are created from the typed data. The existing implementation applies the defined analysis masks, marks country versus non-country partner labels and adds quality flags for missing quantity, zero value and unusually large trade values. It also records partial-year status and saves versioned outputs.

**Safeguard:** Keep source records and the audit trail. Filtering or flagging must be distinguished from correcting a reported value. These transformations do not establish that flagged observations are wrong.


In [ ]:
# 1. Create working copies from the typed copies, using the analysis masks from C08
commodity_imports_clean = commodity_imports_typed[imp_mask].copy()
commodity_exports_clean = commodity_exports_typed[exp_mask].copy()

initial_imp_count = len(commodity_imports_clean)
initial_exp_count = len(commodity_exports_clean)

print("Working copies initialized.")
print(f"Import records kept by the analysis mask: {initial_imp_count:,} of {len(commodity_imports_typed):,}")
print(f"Export records kept by the analysis mask: {initial_exp_count:,} of {len(commodity_exports_typed):,}")

In [ ]:
# 2. Add is_country flag (uses is_non_country from C06)
def assign_geography_flags(df):
    out = df.copy()
    out["is_country"] = [not is_non_country(d, c) for d, c in zip(out["partnerDesc"], out["partnerCode"])]
    return out

commodity_imports_clean = assign_geography_flags(commodity_imports_clean)
commodity_exports_clean = assign_geography_flags(commodity_exports_clean)

print("Imports Geography Distribution (is_country):")
print(commodity_imports_clean['is_country'].value_counts())
print("\nExports Geography Distribution (is_country):")
print(commodity_exports_clean['is_country'].value_counts())

In [ ]:
# 3. Apply Quality and Value Anomaly Flags (unchanged)
def apply_quality_flags(df):
    df_flagged = df.copy()

    # Flag missing or zero quantities
    df_flagged['flag_missing_qty'] = (df_flagged['qty'].isnull()) | (df_flagged['qty'] == 0)

    # Flag zero values
    df_flagged['flag_zero_value'] = (df_flagged['primaryValue'].isnull()) | (df_flagged['primaryValue'] == 0)

    # Flag extreme outliers using standard IQR (Interquartile Range) boundary of 3x IQR
    q25 = df_flagged['primaryValue'].quantile(0.25)
    q75 = df_flagged['primaryValue'].quantile(0.75)
    iqr = q75 - q25
    extreme_threshold = q75 + (3.0 * iqr)

    df_flagged['flag_extreme_value'] = df_flagged['primaryValue'] > extreme_threshold
    return df_flagged

commodity_imports_clean = apply_quality_flags(commodity_imports_clean)
commodity_exports_clean = apply_quality_flags(commodity_exports_clean)

print("Imports Missing/Zero Qty Count:", commodity_imports_clean['flag_missing_qty'].sum())
print("Imports Extreme Value Count:", commodity_imports_clean['flag_extreme_value'].sum())
print("\nExports Missing/Zero Qty Count:", commodity_exports_clean['flag_missing_qty'].sum())
print("Exports Extreme Value Count:", commodity_exports_clean['flag_extreme_value'].sum())

### Visual 2. Quality-screening flags

**RQ1–RQ2:** Share of working-copy records flagged for missing/zero quantity, zero trade value or extreme trade value. These are screening flags, not confirmed errors.

In [ ]:
flag_cols = ['flag_missing_qty','flag_zero_value','flag_extreme_value']
flag_rates = pd.DataFrame({
    'Imports': commodity_imports_clean[flag_cols].fillna(False).mean().mul(100),
    'Exports': commodity_exports_clean[flag_cols].fillna(False).mean().mul(100)
})
ax = flag_rates.plot(kind='barh', figsize=(10, 4.2))
ax.set(title='Quality flags in analytical working copies', xlabel='Records flagged (%)', ylabel='Screening condition')
plt.tight_layout(); plt.show()
display(flag_rates.round(2))


In [ ]:
# 4. Partial-year flag (uses add_year_flag from C10)
commodity_imports_clean = add_year_flag(commodity_imports_clean)
commodity_exports_clean = add_year_flag(commodity_exports_clean)
print("Temporal flags added successfully. Partial years:", PARTIAL_YEARS)

In [ ]:
# 5. Save the cleaned datasets as versioned CSV outputs (unchanged)
cleaned_imports_path = DATA_DIR / "commodity_imports_clean_v1.csv"
cleaned_exports_path = DATA_DIR / "commodity_exports_clean_v1.csv"

commodity_imports_clean.to_csv(cleaned_imports_path, index=False)
commodity_exports_clean.to_csv(cleaned_exports_path, index=False)

clean_entry = {
    'Step': '4.0',
    'Action': 'Execute Systematic Data Cleaning & Flagging',
    'Reason': 'Generate country and quality flags (is_country, flag_missing_qty, flag_extreme_value, is_partial_year) and export v1 cleaned baselines without deleting records.',
    'Rows Affected': f"imports: {len(commodity_imports_clean)}, exports: {len(commodity_exports_clean)}",
    'Output File': 'commodity_imports_clean_v1.csv / commodity_exports_clean_v1.csv'
}

audit_log = pd.concat([audit_log, pd.DataFrame([clean_entry])], ignore_index=True)
display(audit_log)

## Phase 5. Verify the preprocessing outputs

**Research link:** Objectives 1–2 and research questions 1–2.

Compare the source and working-copy record counts, inspect the treatment of World and other aggregate partner labels, and record verification results in the audit log.

**Reading the output:** Any change in row count should be attributed to an explicit analysis mask or filtering rule. Do not describe all source rows as retained unless the before-and-after table supports that claim.


In [ ]:
# 1. Programmatic Before/After comparison summaries
def generate_comparison_report(raw_df, clean_df, name):
    summary = pd.DataFrame({
        'Metric': [
            'Total Rows',
            'Total Columns',
            'Missing primaryValue',
            'Zero/Null Qty Rows',
            'Unique Partners',
            'Unique Years',
            'Exact Duplicates (excl. metadata)'
        ],
        'Before (Raw)': [
            len(raw_df),
            len(raw_df.columns),
            raw_df['primaryValue'].isnull().sum(),
            ((raw_df['qty'].isnull()) | (raw_df['qty'] == 0)).sum(),
            raw_df['partnerDesc'].nunique(),
            raw_df['refYear'].nunique(),
            raw_df.drop(columns=['source_file', 'source_row'], errors='ignore').duplicated().sum()
        ],
        'After (Cleaned)': [
            len(clean_df),
            len(clean_df.columns),
            clean_df['primaryValue'].isnull().sum(),
            clean_df['flag_missing_qty'].sum(),
            clean_df['partnerDesc'].nunique(),
            clean_df['refYear'].nunique(),
            clean_df.drop(columns=['source_file', 'source_row', 'is_country', 'flag_missing_qty', 'flag_zero_value', 'flag_extreme_value', 'is_partial_year'], errors='ignore').duplicated().sum()
        ]
    })

    # Calculate change
    summary['Absolute Change'] = summary['After (Cleaned)'] - summary['Before (Raw)']

    print(f"\n{'='*15} BEFORE / AFTER REPORT: {name} {'='*15}")
    display(summary)
    return summary

report_imports = generate_comparison_report(commodity_imports_typed, commodity_imports_clean, 'Commodity Imports')
report_exports = generate_comparison_report(commodity_exports_typed, commodity_exports_clean, 'Commodity Exports')

In [ ]:
# 2. Repeat the World and total-versus-commodity checks on clean datasets using 'is_country' flag
print("\n--- Cleaned Data Verification: WORLD Aggregate isolation ---")
# Confirm that when is_country is True, no aggregate regions are included
for name, df in [('Imports', commodity_imports_clean), ('Exports', commodity_exports_clean)]:
    countries_only = df[df['is_country'] == True]
    aggregates_only = df[df['is_country'] == False]

    print(f"\nDataset: {name}")
    print(f"  - Countries-only unique partners: {countries_only['partnerDesc'].nunique()}")
    print(f"  - Isolated aggregates unique partners: {aggregates_only['partnerDesc'].unique()}")
    print(f"  - Combined row check: {len(countries_only) + len(aggregates_only)} (Expected: {len(df)})")

In [ ]:
# 3. Log the audit verification step in our central Analytical Audit Log
check_entry = {
    'Step': '5.0',
    'Action': 'Perform Before/After Cleaning Audit Checks',
    'Reason': 'Generate dataset metrics summary report, verify 100% record retention, and validate correctness of the analytical flags.',
    'Rows Affected': f"imports: {len(commodity_imports_clean)}, exports: {len(commodity_exports_clean)}",
    'Output File': 'Diagnostic Summaries Only'
}

audit_log = pd.concat([audit_log, pd.DataFrame([check_entry])], ignore_index=True)
display(audit_log)

## Phase 6. Explore trade patterns and select illustrative commodities

**Research link:** Objectives 2–3 and research questions 2–3.

This exploratory analysis examines annual imports and exports, the reported trade balance, country-partner rankings, concentration measures, selected HS4 commodities and unit-value screening. The charts below visualise the existing calculations rather than replacing them.

**Comparability safeguards:** World aggregates and country-partner totals are shown separately; partial years are marked; trade values are not counts of shipments; and a unit-value flag does not prove mispricing or error.

**Scope note:** The EDA notebook's historical decision-test implementation uses HS `2710`, `1001` and `8703`, whereas the later Trade Decision Lab prototype also assesses `8517` and `0303`. These are related but not identical analytical scopes. Do not present the three-commodity EDA sensitivity grid as if it covered all five prototype commodities.


In [ ]:
import numpy as np
import pandas as pd

# 1. Nigeria-wide overview: annual totals and trade balance
# Totals come from the World rows in total_trade. Country-only sums are shown beside them.
tt = cast_trade_types(total_trade_raw)
if tt["cmdCode"].str.upper().eq("TOTAL").any():
    tt = tt[tt["cmdCode"].str.upper() == "TOTAL"]
tt["is_world"] = pd.to_numeric(tt["partnerCode"], errors="coerce") == 0
tt["non_country"] = [is_non_country(d, c) for d, c in zip(tt["partnerDesc"], tt["partnerCode"])]
tt["flow"] = tt["flowDesc"].str.lower().str.strip()

worlds_per_cell = tt[tt["is_world"]].groupby(["refYear", "flow"]).size()
print("Most World rows in one year and flow (expect 1):", worlds_per_cell.max())

world_tot = tt[tt["is_world"]].pivot_table(index="refYear", columns="flow", values="primaryValue", aggfunc="sum")
country_tot = tt[~tt["non_country"]].pivot_table(index="refYear", columns="flow", values="primaryValue", aggfunc="sum")

overview_df = pd.DataFrame({
    "Imports_World_USD": world_tot.get("import"),
    "Exports_World_USD": world_tot.get("export"),
    "Imports_country_partners_USD": country_tot.get("import"),
    "Exports_country_partners_USD": country_tot.get("export"),
})
overview_df["Trade_Balance_USD"] = overview_df["Exports_World_USD"] - overview_df["Imports_World_USD"]
overview_df["Imports_non_country_share_%"] = ((1 - overview_df["Imports_country_partners_USD"] / overview_df["Imports_World_USD"]) * 100).round(2)
overview_df["Exports_non_country_share_%"] = ((1 - overview_df["Exports_country_partners_USD"] / overview_df["Exports_World_USD"]) * 100).round(2)
overview_df["Is_Partial_Year"] = overview_df.index.isin(PARTIAL_YEARS)
overview_df = overview_df.reset_index()

print("=== NIGERIA ANNUAL TOTALS (World rows) AND BALANCE ===")
display(overview_df)
print("Partial years (from C10):", PARTIAL_YEARS)
# If the first print says more than 1, there are several World rows per year and flow. Inspect before using the totals.

### Visual 3. Nigeria’s reported annual trade

**RQ2–RQ3:** World-aggregate import and export values by year, with the trade balance shown separately. Partial years, where marked in the analysis, should not be interpreted as directly comparable full-year totals.

In [ ]:
trade_plot = overview_df.copy()
trade_plot['refYear'] = pd.to_numeric(trade_plot['refYear'], errors='coerce')
trade_plot = trade_plot.sort_values('refYear')
fig, ax = plt.subplots(figsize=(11, 4.7))
ax.plot(trade_plot['refYear'], trade_plot['Imports_World_USD'] / 1e9, marker='o', label='Imports')
ax.plot(trade_plot['refYear'], trade_plot['Exports_World_USD'] / 1e9, marker='o', label='Exports')
for _, r in trade_plot.loc[trade_plot['Is_Partial_Year'].fillna(False)].iterrows():
    ax.axvspan(r['refYear']-.35, r['refYear']+.35, alpha=.12)
ax.set(title='Nigeria reported trade by year (World aggregates)', xlabel='Reference year', ylabel='Trade value (US$ billion)')
ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.bar(trade_plot['refYear'].astype(str), trade_plot['Trade_Balance_USD']/1e9)
ax.axhline(0, linewidth=.8, color='black')
ax.set(title='Reported trade balance: exports minus imports', xlabel='Reference year', ylabel='Balance (US$ billion)')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()


In [ ]:
# 2. Country partners for imports (country-only rows from the cleaned data)
countries_imp = commodity_imports_clean[commodity_imports_clean['is_country'] == True]
countries_exp = commodity_exports_clean[commodity_exports_clean['is_country'] == True]

total_imp_all_years = countries_imp['primaryValue'].sum()
partner_summary = countries_imp.groupby('partnerDesc')['primaryValue'].sum().reset_index()
partner_summary['Share_%'] = (partner_summary['primaryValue'] / total_imp_all_years) * 100
partner_summary = partner_summary.sort_values(by='primaryValue', ascending=False).reset_index(drop=True)

print("=== TOP 10 IMPORT TRADING PARTNERS (country partners, all years) ===")
display(partner_summary.head(10))

# Concentration for the latest complete year. Report the numbers only; do not label them "competitive" etc.
year_target = LATEST_YEAR
year_imp = countries_imp[countries_imp['refYear'] == year_target]
year_total = year_imp['primaryValue'].sum()

if year_total > 0:
    partner_shares = year_imp.groupby('partnerDesc')['primaryValue'].sum() / year_total * 100
    hhi = (partner_shares ** 2).sum()
    cr5 = partner_shares.sort_values(ascending=False).head(5).sum()
    print(f"\nTrade concentration for {year_target} (country partners only):")
    print(f"  - Top-5 partner share (CR5): {cr5:.2f}%")
    print(f"  - Herfindahl-Hirschman Index (HHI): {hhi:.2f}")

### Visual 4. Leading import partners

**RQ3:** Country-partner ranking across the available years, using the existing country-only working copy. These are aggregated reported trade values, not shipment counts.

In [ ]:
top_partners = partner_summary.head(10).sort_values('primaryValue')
ax = top_partners.plot.barh(x='partnerDesc', y='primaryValue', figsize=(10, 5), legend=False)
ax.set(title='Top 10 import partners (country-only records)', xlabel='Reported trade value (US$)', ylabel='Partner')
plt.tight_layout(); plt.show()


In [ ]:
# 3. Decision-test commodity selection (rule-based)
# Rule: top imported products (4-digit HS) by value over the latest three complete years
complete_years = LATEST_3
recent_imports = countries_imp[countries_imp['refYear'].isin(complete_years)]

top_hs4 = recent_imports.groupby(['cmdCode', 'cmdDesc'])['primaryValue'].sum().reset_index()
top_hs4 = top_hs4.sort_values(by='primaryValue', ascending=False).head(15)   # top 15 so that failures can be replaced

print("=== EVALUATING COMMODITY SELECTION RULES ===")
selected_commodities = []

for idx, row in top_hs4.iterrows():
    code = row['cmdCode']
    desc = row['cmdDesc']

    product_data = recent_imports[recent_imports['cmdCode'] == code]

    # Check 1: appears in all usable years
    years_present = product_data['refYear'].nunique()
    has_all_years = (years_present == len(complete_years))

    # Check 2: at least 8 country partners in the latest complete year
    partners_latest = product_data[product_data['refYear'] == LATEST_YEAR]['partnerDesc'].nunique()
    has_min_partners = (partners_latest >= 8)

    # Check 3: quantity/weight available (less than 90% missing weight entries)
    missing_weight_pct = (product_data['netWgt'].isnull() | (product_data['netWgt'] == 0)).mean() * 100
    has_weight = (missing_weight_pct < 90.0)

    passed = has_all_years and has_min_partners and has_weight
    status = "PASS" if passed else "FAIL"

    print(f"HS4 {code} ({str(desc)[:40]}...): {status}")
    print(f"  - Appears in all years: {has_all_years} ({years_present}/{len(complete_years)})")
    print(f"  - Partners in {LATEST_YEAR}: {partners_latest} (Required >= 8)")
    print(f"  - Missing Net Weight: {missing_weight_pct:.1f}% (Required < 90%)")

    if passed:
        selected_commodities.append({
            'cmdCode': code,
            'cmdDesc': desc,
            'Total_Value_USD': row['primaryValue']
        })

selected_commodities_df = pd.DataFrame(selected_commodities).head(5)
print("\n=== FINAL COMMODITY SELECTION LIST ===")
display(selected_commodities_df)

### Visual 5. Commodity selection and trade value

**RQ3:** Relative trade values of the commodities that pass the existing selection rules. This chart does not change which commodities the code selects.

In [ ]:
if not selected_commodities_df.empty:
    selection_plot = selected_commodities_df.copy().sort_values('Total_Value_USD')
    selection_plot['Label'] = selection_plot['cmdCode'].astype(str) + ' · ' + selection_plot['cmdDesc'].astype(str).str.slice(0, 32)
    ax = selection_plot.plot.barh(x='Label', y='Total_Value_USD', figsize=(10, max(3.5, len(selection_plot)*.65)), legend=False)
    ax.set(title='Commodities passing the EDA selection rules', xlabel='Reported import value (US$)', ylabel='HS4 commodity')
    plt.tight_layout(); plt.show()
else:
    print('No commodities passed the selection rules; no chart is displayed.')


In [ ]:
# 4. Unit values for the worked example (flag rule unchanged: 3x or one third of the yearly median)
print("=== UNIT VALUE DIAGNOSTICS ===")
if not selected_commodities_df.empty:
    target_code = selected_commodities_df.iloc[0]['cmdCode']
    target_desc = selected_commodities_df.iloc[0]['cmdDesc']

    print(f"Worked example commodity: HS4 {target_code} ({target_desc})")

    comm_df = countries_imp[countries_imp['cmdCode'] == target_code].copy()
    comm_df['unit_value'] = np.where(comm_df['netWgt'] > 0, comm_df['primaryValue'] / comm_df['netWgt'], np.nan)
    median_by_year = comm_df.groupby('refYear')['unit_value'].transform('median')

    comm_df['unit_value_ratio'] = comm_df['unit_value'] / median_by_year
    comm_df['flag_unit_value_anomaly'] = (comm_df['unit_value_ratio'] > 3.0) | (comm_df['unit_value_ratio'] < (1.0/3.0))

    flagged_records = comm_df[comm_df['flag_unit_value_anomaly'] == True]
    missing_weight_share = comm_df['netWgt'].isnull().mean() * 100

    print(f"  - Share of records with missing netWgt: {missing_weight_share:.2f}%")
    print(f"  - Outlier unit value rows flagged: {len(flagged_records)} out of {len(comm_df)}")
    display(flagged_records[['refYear', 'partnerDesc', 'primaryValue', 'netWgt', 'unit_value', 'unit_value_ratio']].head(10))

In [ ]:
# 5. Log EDA results
eda_entry = {
    'Step': '6.0',
    'Action': 'Generate National-Level Overview & Perform HS4 Decision Selection',
    'Reason': 'Annual totals from World rows, concentration for the latest complete year, rule-based commodity selection, unit value diagnostics.',
    'Rows Affected': f"Selected {len(selected_commodities_df)} target items for worked analysis.",
    'Output File': 'Diagnostic Summaries Only'
}

audit_log = pd.concat([audit_log, pd.DataFrame([eda_entry])], ignore_index=True)
display(audit_log)

## Phase 7. Test how analytical choices affect partner rankings

**Research link:** Objectives 2–4 and research questions 2–4.

The existing code compares partner rankings under different analytical treatments for the selected illustrative commodities. This tests whether rankings change when aggregate partners or flagged records are handled differently.

**Interpretation:** A changed ranking is evidence of analytical sensitivity, not proof that a treatment is correct or that an importer actually changed its sourcing decision. The notebook's Phase 7 scenarios should not be confused with the full Trade Decision Lab Model B/C assessments.


In [ ]:
# Step 37: Calculate baselines for both scenarios using cleaned country partners only
# We will run this calculation for our top 3 validated Commodities: HS 2710, 1001, and 8703

selected_hs4 = ['2710', '1001', '8703']

# Helper to print ranking
def display_scenario_rankings(df, years, scenario_name):
    print(f"\n{'='*10} BASELINE RANKINGS FOR: {scenario_name} (Years: {years}) {'='*10}")
    for hs_code in selected_hs4:
        # Filter data
        sub_df = df[(df['cmdCode'] == hs_code) & (df['refYear'].isin(years))]

        # Aggregate total value per country partner
        partner_vals = sub_df.groupby('partnerDesc')['primaryValue'].sum().reset_index()
        partner_vals = partner_vals.sort_values(by='primaryValue', ascending=False).reset_index(drop=True)

        # Calculate shares
        total_val = partner_vals['primaryValue'].sum()
        partner_vals['Share_%'] = (partner_vals['primaryValue'] / total_val * 100).round(2) if total_val > 0 else 0

        print(f"\nHS4 {hs_code} - Top 5 Partners:")
        display(partner_vals.head(5))

# Scenario A: Historical Market Analysis (Complete Years 2016-2024, excluding partial 2025)
complete_years_historical = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
display_scenario_rankings(countries_imp, complete_years_historical, "Scenario A (Historical Market Analysis)")

# Scenario B: Sourcing-Market Screening (Last Three Complete Years 2022-2024)
recent_three_years = [2022, 2023, 2024]
display_scenario_rankings(countries_imp, recent_three_years, "Scenario B (Sourcing-Market Screening)")

In [ ]:
# Step 40: Log the Phase 7 Baseline Calculation step into our Master Audit Log
phase7_entry = {
    'Step': '7.0',
    'Action': 'Generate Scenario A & Scenario B Baselines',
    'Reason': 'Calculate the top 5 country rankings and value shares for HS4 2710, 1001, and 8703 to serve as decision testing baselines.',
    'Rows Affected': 'Analyzed 3 core HS4 commodity groups across 9 historical and 3 recent years.',
    'Output File': 'Diagnostic Summaries Only'
}

audit_log = pd.concat([audit_log, pd.DataFrame([phase7_entry])], ignore_index=True)
display(audit_log)

In [ ]:
# Step 38-39: Define and apply treatments, recalculating top-5 rankings for selected commodities

import pandas as pd
import numpy as np

selected_hs4 = ['2710', '1001', '8703']
years_a = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
years_b = [2022, 2023, 2024]

# Let's get the absolute baseline using ALL raw imports (including administrative aggregates like WORLD/NES)
# to illustrate the critical impact of ignoring 'is_country' filters.
def get_top5(df, hs_code, years, filter_outliers=False, filter_missing_weight=False):
    sub_df = df[(df['cmdCode'] == hs_code) & (df['refYear'].isin(years))].copy()

    if filter_outliers and 'flag_unit_value_anomaly' in sub_df.columns:
        sub_df = sub_df[sub_df['flag_unit_value_anomaly'] == False]

    if filter_missing_weight and 'flag_missing_qty' in sub_df.columns:
        sub_df = sub_df[sub_df['flag_missing_qty'] == False]

    partner_vals = sub_df.groupby('partnerDesc')['primaryValue'].sum().reset_index()
    partner_vals = partner_vals.sort_values(by='primaryValue', ascending=False).reset_index(drop=True)
    return list(partner_vals.head(5)['partnerDesc'])

# Initialize comparison log
results_grid = []

for hs in selected_hs4:
    # Scenario A Baselines
    base_a = get_top5(commodity_imports_typed, hs, years_a) # raw baseline (with World/NES)
    clean_a = get_top5(commodity_imports_clean[commodity_imports_clean['is_country'] == True], hs, years_a) # clean country baseline

    # Scenario B Baselines
    base_b = get_top5(commodity_imports_typed, hs, years_b) # raw baseline
    clean_b = get_top5(commodity_imports_clean[commodity_imports_clean['is_country'] == True], hs, years_b) # clean country baseline

    # Apply Treatment 2: Filter Outliers
    # Compute unit values and flag anomalies first
    comm_df = commodity_imports_clean[commodity_imports_clean['is_country'] == True].copy()
    comm_df['unit_value'] = np.where(comm_df['netWgt'] > 0, comm_df['primaryValue'] / comm_df['netWgt'], np.nan)
    median_by_year = comm_df.groupby(['cmdCode', 'refYear'])['unit_value'].transform('median')
    comm_df['unit_value_ratio'] = comm_df['unit_value'] / median_by_year
    comm_df['flag_unit_value_anomaly'] = (comm_df['unit_value_ratio'] > 3.0) | (comm_df['unit_value_ratio'] < (1.0/3.0))

    treat2_b = get_top5(comm_df, hs, years_b, filter_outliers=True)

    # Apply Treatment 3: Exclude missing weight rows
    treat3_b = get_top5(comm_df, hs, years_b, filter_missing_weight=True)

    # Record comparisons for Scenario A
    change_a = set(base_a) != set(clean_a)
    results_grid.append({
        'Commodity': hs,
        'Scenario': 'Scenario A (Hist)',
        'Baseline (Unfiltered Raw)': base_a,
        'Treatment 1 (Country-only Filter)': clean_a,
        'Did Membership Change?': 'YES' if change_a else 'NO',
        'Positional Shifts': sum([1 for x in clean_a if x not in base_a])
    })

    # Record comparisons for Scenario B
    change_b_t1 = set(base_b) != set(clean_b)
    results_grid.append({
        'Commodity': hs,
        'Scenario': 'Scenario B (Sourcing - Clean Geo)',
        'Baseline (Unfiltered Raw)': base_b,
        'Treatment 1 (Country-only Filter)': clean_b,
        'Did Membership Change?': 'YES' if change_b_t1 else 'NO',
        'Positional Shifts': sum([1 for x in clean_b if x not in base_b])
    })

    change_b_t2 = set(clean_b) != set(treat2_b)
    results_grid.append({
        'Commodity': hs,
        'Scenario': 'Scenario B (Sourcing - Outliers Removed)',
        'Baseline (Clean Country)': clean_b,
        'Treatment 2 (No UV Outliers)': treat2_b,
        'Did Membership Change?': 'YES' if change_b_t2 else 'NO',
        'Positional Shifts': sum([1 for x in treat2_b if x not in clean_b])
    })

results_grid_df = pd.DataFrame(results_grid)
print("=== PHASE 7: DATA QUALITY TREATMENT DECISION SENSITIVITY GRID ===")
display(results_grid_df)

In [ ]:
# Step 40: Log the Sensitivity Analysis Results and write the final verdict into the Audit Log
verdict_entry = {
    'Step': '7.1',
    'Action': 'Evaluate Sourcing Decision Sensitivity to Data Quality',
    'Reason': 'Construct sensitivity matrix mapping changes in top-5 supply rankings under country-filtering and unit-value outlier exclusion treatments.',
    'Rows Affected': 'Compared 3 select commodities across 3 distinct treatment pipelines.',
    'Output File': 'Diagnostic Summaries Only'
}

audit_log = pd.concat([audit_log, pd.DataFrame([verdict_entry])], ignore_index=True)
display(audit_log)

In [ ]:
# Display the complete sensitivity analysis results grid from Phase 7
pd.set_option('display.max_colwidth', None)
print("=== COMPREHENSIVE DECISION SENSITIVITY GRID ===")
display(results_grid_df)

### Visual 6. Sensitivity of top-five partner membership

**RQ2–RQ4:** The grid compares partner shortlists under alternative analytical treatments. A change is not evidence that one shortlist is correct or that an actual sourcing decision changed.

In [ ]:
if not results_grid_df.empty and {'Commodity','Scenario','Did Membership Change?'}.issubset(results_grid_df.columns):
    sensitivity_view = results_grid_df[['Commodity','Scenario','Did Membership Change?']].copy()
    sensitivity_view['Changed'] = sensitivity_view['Did Membership Change?'].astype(str).str.upper().eq('YES').astype(int)
    matrix = sensitivity_view.pivot_table(index='Commodity', columns='Scenario', values='Changed', aggfunc='max')
    fig, ax = plt.subplots(figsize=(max(9, len(matrix.columns)*2.2), max(3, len(matrix)*.7+1)))
    im = ax.imshow(matrix.fillna(-1).to_numpy(), cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(len(matrix.columns)), labels=matrix.columns, rotation=35, ha='right')
    ax.set_yticks(range(len(matrix.index)), labels=matrix.index)
    for r in range(len(matrix.index)):
        for c in range(len(matrix.columns)):
            value = matrix.iloc[r,c]
            ax.text(c,r,'N/A' if pd.isna(value) else ('Changed' if value else 'Stable'),ha='center',va='center',fontsize=8)
    ax.set_title('Top-five partner membership sensitivity by scenario')
    plt.tight_layout(); plt.show()
    display(sensitivity_view)
else:
    print('Sensitivity grid is empty or has unexpected columns; see the original results table.')


## Phase 8. Consolidate evidence and hand over the datasets

**Research link:** Objectives 1–5 and research questions 1–5.

The issue register, versioned data outputs, audit trail and sensitivity results document the evidence available to the Trade Decision Lab prototype. The prototype itself is a separate implementation and should be evaluated on its own functionality.

**Before citing a numerical finding:** Confirm it against the current generated tables, check its denominator and time period, and state whether it describes source data, a screening flag or an analytical treatment.


In [ ]:
# Step 41: Finalise and output the Comprehensive Issue Register
# We gather programmatically verified counts from previous stages to provide complete evidence metadata.

# Identify exact indices/counts where issues occurred
null_qty_count_imp = commodity_imports_clean['flag_missing_qty'].sum()
world_nes_count_imp = (~commodity_imports_clean['is_country']).sum()

# For unit value outliers, recalculate the precise row count for our select commodities
comm_subset = commodity_imports_clean[commodity_imports_clean['is_country'] == True].copy()
comm_subset['unit_value'] = np.where(comm_subset['netWgt'] > 0, comm_subset['primaryValue'] / comm_subset['netWgt'], np.nan)
median_by_year_subset = comm_subset.groupby(['cmdCode', 'refYear'])['unit_value'].transform('median')
comm_subset['unit_value_ratio'] = comm_subset['unit_value'] / median_by_year_subset
comm_subset['flag_unit_value_anomaly'] = (comm_subset['unit_value_ratio'] > 3.0) | (comm_subset['unit_value_ratio'] < (1.0/3.0))
uv_outliers_count = comm_subset[comm_subset['flag_unit_value_anomaly'] == True].shape[0]

final_issues = [
    {
        'ID': 'ISS-001',
        'Dataset': 'Imports/Exports',
        'Quality Dimension': 'Completeness',
        'Check': 'Check missing values in value columns (cifvalue/fobvalue)',
        'Commodity': 'ALL',
        'Finding': 'Over 75% missingness in auxiliary value columns. primaryValue column remains 100% complete.',
        'Affected Rows': '216,371 (Imports), 16,848 (Exports)',
        'Treatment': 'Utilize primaryValue for all transactional values and net trade balances.',
        'Effect on Results': 'Guarantees 100% coverage; avoids arbitrary null drops or inaccurate zero-fills.',
        'Status': 'Resolved',
        'Remaining Uncertainty': 'None for primaryValue; auxiliary value columns remain unusable.'
    },
    {
        'ID': 'ISS-002',
        'Dataset': 'Imports/Exports',
        'Quality Dimension': 'Consistency (Geographic)',
        'Check': 'Isolate aggregate regional and administrative entities (WORLD, NES, etc.)',
        'Commodity': 'ALL',
        'Finding': 'Administrative double-counting aggregates (e.g., WORLD, NES) exist within partnerDesc.',
        'Affected Rows': f"{world_nes_count_imp} rows (Imports)",
        'Treatment': "Apply boolean country-only filter ('is_country' == True).",
        'Effect on Results': "Critical. Excludes 'WORLD' from partner ranking, enabling authentic partner country rankings.",
        'Status': 'Resolved',
        'Remaining Uncertainty': 'Minor unspecified trade remains in regional NES categories.'
    },
    {
        'ID': 'ISS-003',
        'Dataset': 'Imports/Exports',
        'Quality Dimension': 'Completeness (Metrics)',
        'Check': 'Compare positive primaryValue with zero/null quantities and weights',
        'Commodity': 'ALL',
        'Finding': 'Positive value records are completely missing quantity or weight measurements.',
        'Affected Rows': f"{null_qty_count_imp} rows (Imports)",
        'Treatment': "Flag records using 'flag_missing_qty'; analyze volume-sensitive indicators separately.",
        'Effect on Results': 'Preserves financial metrics in aggregate rankings while preventing faulty volume-based calculations.',
        'Status': 'Resolved',
        'Remaining Uncertainty': 'Actual traded volume of flagged records cannot be determined from metadata.'
    },
    {
        'ID': 'ISS-004',
        'Dataset': 'Imports (Refined)',
        'Quality Dimension': 'Accuracy (Unit Value)',
        'Check': 'Unit value outlier deviation (exceeding 3x or falling below 1/3 of annual median)',
        'Commodity': '2710, 1001, 8703',
        'Finding': 'Significant unit value discrepancies exist, distorting physical quantities relative to financial values.',
        'Affected Rows': f"{uv_outliers_count} flagged records (Subset)",
        'Treatment': "Exclude records matching 'flag_unit_value_anomaly' from sourcing evaluations.",
        'Effect on Results': 'Alters Top 5 partner rankings for petroleum and cars, while showing no impact on robust wheat data.',
        'Status': 'Resolved',
        'Remaining Uncertainty': 'Some extreme records might represent actual premium pricing or custom codes rather than raw error.'
    }
]

final_issue_register_df = pd.DataFrame(final_issues)
print("=== FINAL MASTER ISSUE REGISTER ===")
display(final_issue_register_df)

In [ ]:
import os

# Define the output path for the issue register CSV
issue_register_path = DATA_DIR / "comtrade_quality_issue_register_v1.csv"

# Export the DataFrame to CSV
final_issue_register_df.to_csv(issue_register_path, index=False)

# Verify the export
if issue_register_path.exists():
    print(f"Success: Issue Register exported successfully to {issue_register_path}")
    print(f"File size: {os.path.getsize(issue_register_path)} bytes")
else:
    print("Error: Issue Register export failed.")

### Reproducibility and handover notes

The notebook retains the source-row tracing approach, audit log, cleaned/flagged working copies and saved issue register. Keep all generated CSVs alongside the notebook when sharing the research evidence.

**Analytical rules:**

- Exclude World and other aggregate labels from country rankings. Use World totals separately where appropriate.
- Keep commodity codes as strings, especially `0303`.
- Preserve missing-weight and unit-value flags. Excluding flagged observations is a sensitivity treatment, not an automatic correction.
- Distinguish the three HS codes used in this notebook's Phase 7 demonstration from the five used by the final Trade Decision Lab prototype.
- Treat any statement about real import–export decisions as a potential implication, not an observed causal outcome.

The next sections provide a structured reading guide, not new empirical findings.


### Findings guide: what the outputs can establish

**Quality conditions (objective 1):** Use the profiling tables, missingness diagnostics, partner review and issue register to identify observed conditions. Report the field, denominator, period and whether a value is missing, zero or merely flagged.

**Effects on analysis (objectives 2–3):** Use the annual overview, partner tables, concentration measures and Phase 7 sensitivity grid to show where different treatments yield different outputs. Explain which treatment was applied before claiming a ranking changed.

**Decision implications (objective 4):** A change in the top-five partners could affect which markets an analyst investigates first. A missing weight can limit unit-value comparisons. Neither observation proves an actual commercial decision was taken.

**Prototype demonstration (objective 5):** Trade Decision Lab provides a separate interactive demonstration of the data-quality and decision-fitness assessments. This acquisition/EDA notebook supplies provenance and preliminary exploration, not a substitute for the prototype's full outputs.

**Limitations:** Classification changes, potential reporting gaps, incomplete weight data and partner-label ambiguity may constrain interpretation. Do not treat heuristic outlier flags as confirmed errors.


### Research-question alignment and final checks

| Research question | Evidence in this notebook | Boundary |
|---|---|---|
| RQ1. What quality issues exist? | Phases 1–3, profiling and issue register | Screening does not prove error |
| RQ2. How do issues affect analytical results? | Phases 4–7, before/after checks and sensitivity tests | Treatments are not verified corrections |
| RQ3. How much do patterns and rankings change? | Phase 6 charts and Phase 7 ranking grid | Scope is the selected commodities and periods |
| RQ4. What might this mean for decisions? | Interpretation of partner and unit-value changes | No actual decisions were observed |
| RQ5. How can a prototype help? | Saved evidence and linkage to Trade Decision Lab | The app is evaluated separately |

**Presentation guidance:** Cite figures directly from executed tables and charts. State the dataset, years, trade flow, HS code and denominator. Retain the original data-loading code and audit trail so another researcher can reproduce the analysis.
